# Session 5: Data Transformation

**Learning goals:**

- Clean and standardise text with Python and pandas string operations.
- Parse, extract and calculate with dates and times.
- Define reusable functions and apply them to data.
- Convert columns to appropriate data types.
- Encode categorical values.
- Scale numeric values with min–max and standard scaling.

**Prerequisites:**

- Basic Python variables, lists and dictionaries.
- Loading and inspecting a pandas `DataFrame`.
- Selecting columns with square brackets.


## Outline

1. **Set up the data**: create and inspect a deliberately messy customer table
2. **String operations**: clean text with Python methods and pandas `.str`
3. **Datetime operations**: parse dates, extract components and calculate durations
4. **Defining functions**: package repeated transformations into reusable code
5. **Converting data types**: safely convert text into numbers, dates, Boolean values and strings
6. **Encoding and scaling**: encode categorical values and rescale numeric columns
7. **Exercises**: transform a second dataset from start to finish

## 1. Set up the data

Data transformation changes values, formats or structures so that data is ready for analysis. Real datasets often contain inconsistent capitalisation, extra spaces, invalid dates and numbers stored as text.

This notebook uses a small in-memory dataset, so every cell can run without downloading a file.

In [1]:
# Import pandas and use the standard alias.
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
# Create a deliberately messy customer table.
customers_raw = pd.DataFrame(
    {
        "customer_id": [" c-001 ", "C-002", "c-003", " C-004", "c-005 ", "C-006"],
        "full_name": [
            "  mia chen",
            "NOAH WILLIAMS ",
            "olivia  brown",
            "Liam O'Connor",
            "ava nguyen ",
            " ETHAN SMITH",
        ],
        "suburb": ["perth", " FREMANTLE", "Joondalup ", "perth", "fremantle", "JOONDALUP"],
        "signup_date": ["05/01/2025", "17/02/2025", "31/02/2025", None, "09/04/2025", "22/05/2025"],
        "spend_aud": ["$1,250.50", "875", "not recorded", " 2,040.00 ", "590.5", "1,680"],
        "visits": ["12", "8", "five", "15", None, "11"],
        "member": ["Yes", "yes", "No", "Y", "N", "TRUE"],
        "channel": [" Social Media ", "email", "EMAIL", "In-store", "in store", "social-media"],
    }
)

customers_raw

,customer_id,full_name,suburb,signup_date,spend_aud,visits,member,channel
0,c-001,mia chen,perth,05/01/2025,"$1,250.50",12,Yes,Social Media
1,C-002,NOAH WILLIAMS,FREMANTLE,17/02/2025,875,8,yes,email
2,c-003,olivia brown,Joondalup,31/02/2025,not recorded,five,No,EMAIL
3,C-004,Liam O'Connor,perth,NaN,"2,040.00",15,Y,In-store
4,c-005,ava nguyen,fremantle,09/04/2025,590.5,NaN,N,in store
5,C-006,ETHAN SMITH,JOONDALUP,22/05/2025,"1,680",11,TRUE,social-media


Always inspect the table before transforming it. `.info()` shows that every raw column is currently stored as text (`object` or `str`), even when some values represent dates or numbers.

In [3]:
# Inspect shape, column names, missing values and current data types.
print("Rows and columns:", customers_raw.shape)
print("\nMissing values:")
print(customers_raw.isna().sum())
print("\nData types:")
customers_raw.info()

Rows and columns: (6, 8)

Missing values:
customer_id    0
full_name      0
suburb         0
signup_date    1
spend_aud      0
visits         1
member         0
channel        0
dtype: int64

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  6 non-null      str  
 1   full_name    6 non-null      str  
 2   suburb       6 non-null      str  
 3   signup_date  5 non-null      str  
 4   spend_aud    6 non-null      str  
 5   visits       5 non-null      str  
 6   member       6 non-null      str  
 7   channel      6 non-null      str  
dtypes: str(8)
memory usage: 516.0 bytes


In [4]:
# Work on a copy so the original raw data remains available for comparison.
customers = customers_raw.copy()

## 2. String operations

A Python string is an immutable sequence of characters. String methods return a **new** string rather than changing the original value.

Common methods include:

| Method | Purpose | Example result |
| --- | --- | --- |
| `.strip()` | remove leading and trailing whitespace | `"  Perth ".strip()` → `"Perth"` |
| `.lower()` / `.upper()` | change letter case | `"Email".lower()` → `"email"` |
| `.title()` | capitalise words | `"mia chen".title()` → `"Mia Chen"` |
| `.replace(old, new)` | replace matching text | `"in-store".replace("-", " ")` → `"in store"` |
| `.split(separator)` | split one string into a list | `"C-001".split("-")` → `["C", "001"]` |
| `separator.join(items)` | join items into one string | `"-".join(["C", "001"])` → `"C-001"` |

In [5]:
# Python string methods return new values.
raw_text = "  social-media  "
clean_text = raw_text.strip().lower().replace("-", " ")

print("Original:", repr(raw_text))
print("Cleaned: ", repr(clean_text))

Original: '  social-media  '
Cleaned:  'social media'


In [6]:
# split() and join() change between strings and lists of strings.
identifier = "C-001"
parts = identifier.split("-")

print("Parts:", parts)
print("Joined again:", "-".join(parts))

Parts: ['C', '001']
Joined again: C-001


### String operations in pandas

For a pandas `Series`, use the `.str` accessor before the string method. Methods can be chained so that each step feeds into the next one.

In [7]:
# Remove outer whitespace and standardise case.
customers["customer_id"] = (
    customers["customer_id"]
    .str.strip()  # Remove whitespace from both ends of each customer ID.
    .str.upper()  # Convert every customer ID to uppercase.
)
customers["full_name"] = (
    customers["full_name"]
    .str.strip()                           # Remove whitespace from both ends of each name.
    .str.replace(r"\s+", " ", regex=True)  # Replace repeated whitespace with one space.
    .str.title()                           # Convert names to title case.
)
customers["suburb"] = (
    customers["suburb"]
    .str.strip()  # Remove whitespace from both ends of each suburb.
    .str.title()  # Convert suburb names to title case.
)

customers[["customer_id", "full_name", "suburb"]]

,customer_id,full_name,suburb
0,C-001,Mia Chen,Perth
1,C-002,Noah Williams,Fremantle
2,C-003,Olivia Brown,Joondalup
3,C-004,Liam O'Connor,Perth
4,C-005,Ava Nguyen,Fremantle
5,C-006,Ethan Smith,Joondalup


The channel labels describe the same three categories in several ways. Standardise punctuation and spacing first, then use a replacement dictionary for the remaining business rules.

In [8]:
# Standardise channel labels.
customers["channel"] = (
    customers["channel"]
    .str.strip()                           # Remove whitespace from both ends of each channel label.
    .str.lower()                           # Convert channel labels to lowercase.
    .str.replace("-", " ", regex=False)    # Replace hyphens with spaces.
    .str.replace(r"\s+", " ", regex=True)  # Replace repeated whitespace with one space.
    .replace({"in store": "in-store", "social media": "social media"})
)

customers["channel"].value_counts()

channel
social media    2
email           2
in-store        2
Name: count, dtype: int64

Useful string tests include `.str.contains()`, `.str.startswith()` and `.str.endswith()`. They return Boolean values that can be used for filtering.

In [9]:
# Find customers whose cleaned channel contains the word "social".
social_customers = customers[
    customers["channel"].str.contains(
        "social",
        na=False,  # Treat missing channel values as non-matches.
    )
]
social_customers[["customer_id", "full_name", "channel"]]

,customer_id,full_name,channel
0,C-001,Mia Chen,social media
5,C-006,Ethan Smith,social media


## 3. Datetime operations

Python's `datetime` module provides `date`, `datetime` and `timedelta` objects. These are useful for individual values. Pandas extends the same ideas to complete columns.

In [10]:
# Work with individual dates using Python's standard library.
from datetime import date, datetime, timedelta

report_date = date(2025, 6, 30)
one_week_later = report_date + timedelta(days=7)
parsed_example = datetime.strptime("17/02/2025", "%d/%m/%Y")

print("Report date:", report_date)
print("One week later:", one_week_later)
print("Parsed datetime:", parsed_example)
print("Australian display:", parsed_example.strftime("%d/%m/%Y"))

Report date: 2025-06-30
One week later: 2025-07-07
Parsed datetime: 2025-02-17 00:00:00
Australian display: 17/02/2025


### Parsing a pandas datetime column

Use `pd.to_datetime()` to convert strings into pandas datetimes. The format `%d/%m/%Y` states that the values use day/month/year order.

With `errors="coerce"`, impossible or missing dates become `NaT` (Not a Time) instead of stopping the notebook. This is useful for diagnosis, but invalid values should still be investigated.

In [11]:
# Convert the signup date column safely.
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    format="%d/%m/%Y",
    errors="coerce",
)

customers[["customer_id", "signup_date"]]

,customer_id,signup_date
0,C-001,2025-01-05
1,C-002,2025-02-17
2,C-003,NaT
3,C-004,NaT
4,C-005,2025-04-09
5,C-006,2025-05-22


In [12]:
# Count dates that could not be parsed.
invalid_date_count = customers["signup_date"].isna().sum()
print("Missing or invalid signup dates:", invalid_date_count)

Missing or invalid signup dates: 2


Use the pandas `.dt` accessor to extract components from an entire datetime column. For example, `.dt.quarter` returns the calendar quarter as an integer from 1 to 4, while `.dt.to_period("Q")` combines the year and calendar quarter in a value such as `2025Q1`.

In [13]:
# Extract calendar components for analysis.
customers["signup_year"] = customers["signup_date"].dt.year  # Extract the four-digit year.
customers["signup_month"] = customers["signup_date"].dt.month  # Extract the month number.
customers["signup_quarter"] = customers["signup_date"].dt.quarter  # Return quarter 1–4.
customers["signup_year_quarter"] = (
    customers["signup_date"].dt.to_period("Q")  # Combine year and quarter, for example 2025Q1.
)
customers["signup_month_name"] = (
    customers["signup_date"].dt.month_name()  # Return the full month name.
)
customers["signup_weekday"] = (
    customers["signup_date"].dt.day_name()  # Return the full weekday name.
)

customers[
    [
        "signup_date",
        "signup_year",
        "signup_month",
        "signup_quarter",
        "signup_year_quarter",
        "signup_month_name",
        "signup_weekday",
    ]
]

,signup_date,signup_year,signup_month,signup_quarter,signup_year_quarter,signup_month_name,signup_weekday
0,2025-01-05,2025.0,1.0,1.0,2025Q1,January,Sunday
1,2025-02-17,2025.0,2.0,1.0,2025Q1,February,Monday
2,NaT,NaN,NaN,NaN,NaT,NaN,NaN
3,NaT,NaN,NaN,NaN,NaT,NaN,NaN
4,2025-04-09,2025.0,4.0,2.0,2025Q2,April,Wednesday
5,2025-05-22,2025.0,5.0,2.0,2025Q2,May,Thursday


Subtracting datetimes produces a duration. `.dt.days` extracts the number of whole days from that duration.

In [14]:
# Calculate customer tenure at a fixed reporting date.
report_timestamp = pd.Timestamp("2025-06-30")
customers["days_since_signup"] = (report_timestamp - customers["signup_date"]).dt.days

customers[["customer_id", "signup_date", "days_since_signup"]]

,customer_id,signup_date,days_since_signup
0,C-001,2025-01-05,176.0
1,C-002,2025-02-17,133.0
2,C-003,NaT,NaN
3,C-004,NaT,NaN
4,C-005,2025-04-09,82.0
5,C-006,2025-05-22,39.0


## 4. Defining functions

A function groups reusable instructions under a meaningful name. Use `def`, list the inputs as parameters, document the purpose, and use `return` to send a result back.

Functions are particularly helpful when a transformation contains several steps or business rules.

In [15]:
def clean_currency_text(value):
    """Remove common currency formatting while preserving missing values."""
    # Preserve missing values rather than converting them to text.
    if pd.isna(value):
        return value

    # Remove currency symbols, thousands separators and outer whitespace.
    return str(value).replace("$", "").replace(",", "").strip()


# Test the function on one value before applying it to a column.
clean_currency_text("$1,250.50")

'1250.50'

A function should ideally do one clear job. Testing a few representative values—including an unusual one—helps reveal errors early.

In [16]:
# Test normal, missing and non-numeric inputs.
test_values = ["$1,250.50", None, "not recorded"]
[clean_currency_text(value) for value in test_values]

['1250.50', None, 'not recorded']

Use `Series.map()` to transform each value in one pandas `Series`. Its main input can be either a function or a dictionary:

| Input | What `.map()` does | Typical use |
| --- | --- | --- |
| function | passes each Series value to the function and stores the returned result | apply cleaning logic or a calculation |
| dictionary | looks up each Series value as a dictionary key and returns the matching dictionary value | recode known labels or categories |

With a function, use the pattern `series.map(function_name)`. The function is written without parentheses because `.map()` calls it separately for each value.

In [17]:
# Apply the function to each spend value.
customers["spend_text_clean"] = customers["spend_aud"].map(clean_currency_text)

customers[["spend_aud", "spend_text_clean"]]

,spend_aud,spend_text_clean
0,"$1,250.50",1250.50
1,875,875
2,not recorded,not recorded
3,"2,040.00",2040.00
4,590.5,590.5
5,"1,680",1680


With a dictionary, use the pattern `series.map(mapping_dictionary)`. Dictionary keys are the original values and dictionary values are the replacements. A value that is not present as a key becomes missing (`NaN`).

In [18]:
# Map known codes to descriptive labels with a dictionary.
member_codes = pd.Series(["Y", "N", "Y", "Unknown"], name="member_code")
member_lookup = {"Y": "Member", "N": "Not a member"}

mapped_status = member_codes.map(member_lookup)

pd.DataFrame(
    {"member_code": member_codes, "member_status": mapped_status}
)

,member_code,member_status
0,Y,Member
1,N,Not a member
2,Y,Member
3,Unknown,NaN


Functions can also express business rules. Type hints communicate the expected inputs and output, but they do not automatically validate the data.

In [19]:
# Define reusable business rules for spend segments.
def classify_spend(spend: float) -> str:
    """Classify annual spend into a simple customer segment."""
    # Handle missing numeric values before checking thresholds.
    if pd.isna(spend):
        return "Unknown"

    # Test thresholds from highest to lowest.
    if spend >= 1500:
        return "High"
    if spend >= 750:
        return "Medium"
    return "Low"

## 5. Converting data types

Correct data types make calculations reliable and reduce ambiguity.

| Target type | Common pandas approach | Typical use |
| --- | --- | --- |
| numeric | `pd.to_numeric()` or `.astype()` | amounts, counts and measurements |
| datetime | `pd.to_datetime()` | dates and timestamps |
| Boolean | `.astype("boolean")` | yes/no flags |
| string | `.astype("string")` | identifiers and text |

Use `.astype()` when values are already clean. Use `pd.to_numeric(..., errors="coerce")` when some values may be invalid.

In [20]:
# Convert cleaned currency text to floating-point numbers.
customers["spend_aud"] = pd.to_numeric(
    customers["spend_text_clean"],
    errors="coerce",
)

customers[["spend_text_clean", "spend_aud"]]

,spend_text_clean,spend_aud
0,1250.50,1250.5
1,875,875.0
2,not recorded,NaN
3,2040.00,2040.0
4,590.5,590.5
5,1680,1680.0


Pandas nullable integer type `Int64` can store whole numbers and missing values together. Correct the known word `"five"` before converting the visits column.

In [21]:
# Step 1: standardise the known text value.
customers["visits"] = customers["visits"].replace({"five": "5"})

# Step 2: convert invalid values to missing, then use nullable integers.
customers["visits"] = pd.to_numeric(
    customers["visits"],
    errors="coerce",
).astype("Int64")

customers[["customer_id", "visits"]]

,customer_id,visits
0,C-001,12
1,C-002,8
2,C-003,5
3,C-004,15
4,C-005,<NA>
5,C-006,11


Boolean conversion requires an explicit mapping when the raw data uses several labels. Standardise the text first, then map each accepted label.

In [22]:
# Convert varied yes/no labels to pandas nullable Boolean values.
member_map = {
    "yes": True,
    "y": True,
    "true": True,
    "no": False,
    "n": False,
    "false": False,
}

customers["member"] = (
    customers["member"]
    .str.strip()           # Remove outer whitespace.
    .str.lower()           # Standardise labels before dictionary lookup.
    .map(member_map)       # Convert recognised labels to True or False.
    .astype("boolean")  # Preserve missing values with pandas nullable Boolean type.
)

customers[["customer_id", "member"]]

,customer_id,member
0,C-001,True
1,C-002,True
2,C-003,False
3,C-004,True
4,C-005,False
5,C-006,True


After converting `spend_aud` to numeric values, apply the business-rule function from Section 4 to create a text label for each customer.

In [23]:
# Apply the business-rule function after numeric conversion.
customers["spend_segment"] = customers["spend_aud"].map(classify_spend)

customers[["customer_id", "spend_aud", "spend_segment"]]

,customer_id,spend_aud,spend_segment
0,C-001,1250.5,Medium
1,C-002,875.0,Medium
2,C-003,NaN,Unknown
3,C-004,2040.0,High
4,C-005,590.5,Low
5,C-006,1680.0,High


In [24]:
# Confirm that the classification results are stored as text.
print("spend_segment data type:", customers["spend_segment"].dtype)

spend_segment data type: str


Inspect the transformed types and invalid values. Conversion with `errors="coerce"` is safe for execution, but it can hide data-quality problems if the resulting missing values are not checked.

In [25]:
# Review types and conversion results.
print(customers.dtypes)
print("\nMissing values after conversion:")
print(customers[["signup_date", "spend_aud", "visits", "member"]].isna().sum())

customer_id                       str
full_name                         str
suburb                            str
signup_date            datetime64[us]
spend_aud                     float64
visits                          Int64
member                        boolean
channel                           str
signup_year                   float64
signup_month                  float64
signup_quarter                float64
signup_year_quarter     period[Q-DEC]
signup_month_name                 str
signup_weekday                    str
days_since_signup             float64
spend_text_clean                  str
spend_segment                     str
dtype: object

Missing values after conversion:
signup_date    2
spend_aud      1
visits         1
member         0
dtype: int64


The temporary helper column is no longer required. Dropping it keeps the final table focused.

In [26]:
# Remove the intermediate text column.
customers = customers.drop(columns="spend_text_clean")

customers.head()

,customer_id,full_name,suburb,signup_date,spend_aud,visits,member,channel,signup_year,signup_month,signup_quarter,signup_year_quarter,signup_month_name,signup_weekday,days_since_signup,spend_segment
0,C-001,Mia Chen,Perth,2025-01-05,1250.5,12,True,social media,2025.0,1.0,1.0,2025Q1,January,Sunday,176.0,Medium
1,C-002,Noah Williams,Fremantle,2025-02-17,875.0,8,True,email,2025.0,2.0,1.0,2025Q1,February,Monday,133.0,Medium
2,C-003,Olivia Brown,Joondalup,NaT,NaN,5,False,email,NaN,NaN,NaN,NaT,NaN,NaN,NaN,Unknown
3,C-004,Liam O'Connor,Perth,NaT,2040.0,15,True,in-store,NaN,NaN,NaN,NaT,NaN,NaN,NaN,High
4,C-005,Ava Nguyen,Fremantle,2025-04-09,590.5,<NA>,False,in-store,2025.0,4.0,2.0,2025Q2,April,Wednesday,82.0,Low


## 6. Encoding and Scaling

Python data preparation usually treats categorical and numerical transformations as two separate operations:

1. **Encoding** converts categorical labels into a numeric representation.
2. **Scaling** changes the range or distribution of numerical values.

### 6.1 Encoding categorical variables

A mapping dictionary can assign a chosen integer code to each known label. The numbers act as codes, not quantities, so their order and distance should not be interpreted unless the original categories are genuinely ordered. Values not found in the dictionary become missing.

In [27]:
# Define an explicit integer code for each known channel label.
channel_code_map = {
    "email": 0,
    "in-store": 1,
    "social media": 2,
}

# Map labels to codes and use nullable integers for possible missing values.
customers["channel_code"] = (
    customers["channel"]
    .map(channel_code_map)
    .astype("Int64")
)

customers[["customer_id", "channel", "channel_code"]]

,customer_id,channel,channel_code
0,C-001,social media,2
1,C-002,email,0
2,C-003,email,0
3,C-004,in-store,1
4,C-005,in-store,1
5,C-006,social media,2


One-hot encoding avoids assigning an artificial order. It creates one binary column for each categorical label: `1` indicates that the row belongs to the label, while `0` indicates that it does not. Pandas provides `pd.get_dummies()` for this operation.

In [28]:
# One-hot encode the channel labels as 0/1 columns.
channel_encoded = pd.get_dummies(
    customers["channel"],
    prefix="channel",
    dtype="int64",
)

pd.concat(
    [customers[["customer_id", "channel"]], channel_encoded],
    axis=1,
)

,customer_id,channel,channel_email,channel_in-store,channel_social media
0,C-001,social media,0,0,1
1,C-002,email,1,0,0
2,C-003,email,1,0,0
3,C-004,in-store,0,1,0
4,C-005,in-store,0,1,0
5,C-006,social media,0,0,1


### 6.2 Scaling numerical variables

Scaling helps compare numerical variables measured in different units. Two common methods are:

| Method | Formula | Interpretation |
| --- | --- | --- |
| Min–max scaling | `(x - min) / (max - min)` | values usually range from 0 to 1 |
| Standard scaling | `(x - mean) / standard deviation` | 0 is the mean; magnitude is measured in standard deviations |

When building a predictive model, calculate scaling parameters from the training data only, then reuse those parameters for validation and test data.

In [36]:
# Define reusable min–max scaling.
def min_max_scale(series: pd.Series) -> pd.Series:
    """Scale non-missing values to the range 0–1."""
    # Calculate the parameters used by the scaling formula.
    minimum = series.min()
    spread = series.max() - minimum

    # Avoid division by zero when all values are equal or missing.
    if pd.isna(spread) or spread == 0:
        return pd.Series(pd.NA)

    # Apply (value - minimum) / (maximum - minimum).
    return (series - minimum) / spread


# Scale customer spend while retaining missing values.
customers["spend_min_max_scaled"] = min_max_scale(customers["spend_aud"])
customers[["customer_id", "spend_aud", "spend_min_max_scaled"]]

,customer_id,spend_aud,spend_min_max_scaled
0,C-001,1250.5,0.455329
1,C-002,875.0,0.196275
2,C-003,NaN,NaN
3,C-004,2040.0,1.000000
4,C-005,590.5,0.000000
5,C-006,1680.0,0.751638


Standard scaling subtracts the mean and divides by the standard deviation. The resulting values are z-scores. This example uses the population standard deviation (`ddof=0`), so the scaled non-missing values have a mean of 0 and a population standard deviation of 1.

In [37]:
# Define reusable standard scaling.
def standard_scale(series: pd.Series) -> pd.Series:
    """Scale non-missing values using the population standard deviation."""
    # ddof=0 calculates the population standard deviation.
    standard_deviation = series.std(ddof=0)

    # Avoid division by zero when all values are equal or missing.
    if pd.isna(standard_deviation) or standard_deviation == 0:
        return pd.Series(pd.NA)

    # Centre values on the mean, then divide by the standard deviation.
    return (series - series.mean()) / standard_deviation


# Standard scale customer spend while retaining missing values.
customers["spend_standard_scaled"] = standard_scale(customers["spend_aud"])
customers[["customer_id", "spend_aud", "spend_standard_scaled"]]

,customer_id,spend_aud,spend_standard_scaled
0,C-001,1250.5,-0.069917
1,C-002,875.0,-0.785279
2,C-003,NaN,NaN
3,C-004,2040.0,1.434154
4,C-005,590.5,-1.327278
5,C-006,1680.0,0.748320


In [31]:
# Check the expected properties of the scaled columns.
print("Min–max scaled minimum:", customers["spend_min_max_scaled"].min())
print("Min–max scaled maximum:", customers["spend_min_max_scaled"].max())
print("Standard scaled mean:", round(customers["spend_standard_scaled"].mean(), 10))
print("Standard scaled population standard deviation:", customers["spend_standard_scaled"].std(ddof=0))

Min–max scaled minimum: 0.0
Min–max scaled maximum: 1.0
Standard scaled mean: -0.0
Standard scaled population standard deviation: 0.9999999999999999


## 7. Worked exercise

The `orders_raw` table contains inconsistent product names, Australian-formatted dates, currency symbols and quantities stored as text.

Transform it so that:
1. product names use title case with single spaces
2. `order_date` is a datetime column
3. `quantity` is a nullable integer
4. `unit_price_aud` is numeric
5. `order_total_aud` equals quantity × unit price
6. `order_total_min_max_scaled` contains min–max scaled order totals

In [ ]:
# Exercise data
orders_raw = pd.DataFrame(
    {
        "product": ["  reusable bottle", "NOTEBOOK ", "desk   lamp", "reusable BOTTLE"],
        "order_date": ["03/06/2025", "07/06/2025", "12/06/2025", "18/06/2025"],
        "quantity": ["2", "5", "1", "3"],
        "unit_price_aud": ["$24.95", "$6.50", "$48.00", "$24.95"],
    }
)

orders_raw

In [ ]:
# Exercise scaffold: complete each transformation.
orders = orders_raw.copy()

# orders["product"] = ...
# orders["order_date"] = ...
# orders["quantity"] = ...
# orders["unit_price_aud"] = ...
# orders["order_total_aud"] = ...
# orders["order_total_min_max_scaled"] = ...

orders